# BiteTracker: training experiment in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/misskanyi/BiteTracker-ML/blob/main/notebooks/bitetracker_training.ipynb)

This notebook is part of the Sprint II development workflow:

**Local IDE → GitHub → Google Colab → Google Drive → Local IDE + AI agent**

1. Code is written locally and pushed to GitHub.
2. This notebook clones an exact commit from GitHub into Colab.
3. It runs the project's existing pipeline (`scripts/train_models.py`). No training code is copied into this notebook.
4. It saves the model, metrics, plots and logs to a new folder in Google Drive: `MyDrive/BiteTracker/experiments/run_YYYYMMDD_HHMMSS/`.
5. Google Drive for desktop syncs that folder to the Mac, where the outputs can be reviewed in the IDE.

**How to use:** check the settings in section 1, then choose **Runtime → Run all**. Colab will ask for permission to use your Google Drive and your Colab Secrets.

> **Only committed and pushed code runs here.** Colab clones from GitHub, so local changes that haven't been committed and pushed are not included. The notebook records the exact commit SHA of every run.

> **Methodology status.** The real-data (Food4Education) pipeline still has open issues: target outlier capping and validation leakage. Real-data results from this notebook are labelled **PROVISIONAL** and are not final research findings. The default data source is the real **Food4Education** sheet; the synthetic dataset is kept only for end-to-end testing.

## 1. Project setup

Edit the settings below if needed. The defaults train on the real Food4Education data; set `DATA_SOURCE = "synthetic"` only for a quick end-to-end test.

In [ ]:
# ---- Experiment settings (edit these) ---------------------------------------
REPO = "misskanyi/BiteTracker-ML"      # GitHub repository (may be private)
BRANCH = "main"                     # branch to clone
COMMIT = ""                            # optional: exact commit SHA to run; blank = latest commit on BRANCH
DATA_SOURCE = "food4education"         # "food4education" (real data, PROVISIONAL) or "synthetic" (end-to-end test only)
RANDOM_SEED = 42                       # passed to the pipeline as RANDOM_SEED
SAVE_ROW_LEVEL_OUTPUTS = None          # None = automatic: yes for synthetic, no for Food4Education
DRIVE_EXPERIMENTS_DIR = "BiteTracker/experiments"  # folder inside MyDrive
# ------------------------------------------------------------------------------

import os
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

assert DATA_SOURCE in {"synthetic", "food4education"}, "DATA_SOURCE must be 'synthetic' or 'food4education'"
IN_COLAB = "google.colab" in sys.modules
STARTED_AT = datetime.now(timezone.utc)

# Outside Colab (e.g. a local test run) these environment variables can redirect the
# clone source and the output folder; in Colab the defaults are used.
REPO_URL = os.getenv("BITETRACKER_NOTEBOOK_REPO_URL", f"https://github.com/{REPO}.git")
if IN_COLAB:
    WORK_DIR = Path("/content/BiteTracker-ML")
else:
    WORK_DIR = Path(tempfile.mkdtemp(prefix="bitetracker_colab_")) / "BiteTracker-ML"

print(f"Running in Google Colab: {IN_COLAB}")
print(f"Python {sys.version.split()[0]}")
print(f"Repository: {REPO} (branch {BRANCH}{', commit ' + COMMIT if COMMIT else ''})")
print(f"Data source: {DATA_SOURCE}")
print(f"Clone folder: {WORK_DIR}")
print(f"Started (UTC): {STARTED_AT.isoformat(timespec='seconds')}")

## 2. GitHub repository access

The repository is private, so cloning it needs a read-only GitHub token stored as a **Colab Secret**. Never paste the token into a cell.

One-time setup:
1. On GitHub, go to **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**.
   Under *Repository access*, choose *Only select repositories* → `misskanyi/BiteTracker-ML`. Under *Permissions*, set *Contents* to **Read-only**. Set an expiry date.
2. In Colab, click the **key icon (Secrets)** in the left sidebar, then **Add new secret**. Name it `GITHUB_TOKEN`, paste the token as the value, and turn on **Notebook access**.

The token is passed to `git` as a one-off request header. It is not printed, not written to `.git/config`, and not saved with the experiment.

In [ ]:
import base64
import shutil


def _secret(name):
    """Read a Colab Secret (or an environment variable outside Colab); None if not set."""
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name) or None
        except Exception:  # secret missing, or notebook access not granted
            return None
    return os.getenv(name) or None


def _run_git(args, token=None, cwd=None):
    """Run git without prompting; the token (if any) is sent as a header and scrubbed from output."""
    cmd = ["git"]
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd += ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}"]
    result = subprocess.run(cmd + args, cwd=cwd, capture_output=True, text=True,
                            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***").replace(basic, "***")
    return result.returncode, output


if WORK_DIR.exists() and IN_COLAB:
    shutil.rmtree(WORK_DIR)  # always start from a fresh clone so the run matches the commit exactly

_token = _secret("GITHUB_TOKEN")
print("GitHub token: found in Colab Secrets" if _token else "GitHub token: not set (works only if the repository is public)")
code_, out = _run_git(["clone", "--quiet", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORK_DIR)], token=_token)
if code_ != 0:
    raise RuntimeError(
        "Could not clone the repository.\n" + out + "\n\n"
        "If the repository is private, add a read-only GitHub token as the Colab Secret 'GITHUB_TOKEN' "
        "(see the instructions above), give this notebook access to it, and re-run this cell. "
        f"Also check that the branch '{BRANCH}' has been pushed to GitHub."
    )
if COMMIT:
    code_, out = _run_git(["fetch", "--quiet", "origin", COMMIT], token=_token, cwd=WORK_DIR)
    code_, out = _run_git(["checkout", "--quiet", COMMIT], cwd=WORK_DIR)
    if code_ != 0:
        raise RuntimeError(f"Commit {COMMIT} not found on GitHub. Has it been pushed?\n{out}")
del _token

GIT_SHA = subprocess.run(["git", "rev-parse", "HEAD"], cwd=WORK_DIR, capture_output=True, text=True).stdout.strip()
print(f"Cloned commit: {GIT_SHA}")
print(subprocess.run(["git", "log", "-1", "--format=%h %cI %s"], cwd=WORK_DIR, capture_output=True, text=True).stdout.strip())

# This notebook needs files that only exist once the Sprint II changes are committed and pushed.
needed = ["scripts/train_models.py", "src/experiment_tracking.py", "data/sample/synthetic_school_meals.csv"]
if DATA_SOURCE == "food4education":
    needed.append("src/google_sheet_loader.py")
missing = [p for p in needed if not (WORK_DIR / p).exists()]
if missing:
    raise RuntimeError(
        f"Commit {GIT_SHA[:7]} does not contain: {missing}.\n"
        "Commit and push the local changes (including this notebook's helper modules) and re-run."
    )

## 3. Dependency installation

Installs the pinned ML package versions from `requirements-colab.txt`. These match `requirements.txt`, so a model saved here loads locally with the same library versions. The Django web packages are not installed because this notebook doesn't run the website.

If this cell asks you to **restart the session**, choose **Runtime → Restart session** and then **Run all** again.

In [ ]:
import importlib.metadata as md

req_file = WORK_DIR / "requirements-colab.txt"
if not req_file.exists():
    req_file = WORK_DIR / "requirements.txt"
pip = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(req_file)],
                     capture_output=True, text=True)
PIP_LOG = pip.stdout + pip.stderr
if pip.returncode != 0:
    print(PIP_LOG[-3000:])
    raise RuntimeError("pip install failed (see the output above).")

pins = dict(line.split("==") for line in req_file.read_text().splitlines()
            if "==" in line and not line.startswith("#"))
mismatched = {name: (want, md.version(name)) for name, want in pins.items() if md.version(name) != want}
if mismatched:
    raise RuntimeError(f"Installed versions differ from the pins: {mismatched}")

# Colab may have imported older numpy/pandas before the install; those stay in memory until a restart.
stale = [m for m, dist in (("numpy", "numpy"), ("pandas", "pandas"))
         if m in sys.modules and getattr(sys.modules[m], "__version__", None) != md.version(dist)]
if stale:
    raise RuntimeError(f"{stale} were loaded before the install. Use Runtime -> Restart session, then Run all.")

if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    print(f"Note: the project is tested on Python 3.10-3.12; this runtime has {sys.version.split()[0]}.")

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

from src import experiment_tracking as et  # noqa: E402

for name, version in et.package_versions().items():
    print(f"{name:15s} {version}")

## 4. Google Drive mounting

Colab asks you to sign in and allow access to your Google Drive. Each run is saved to a **new** folder under `MyDrive/BiteTracker/experiments/`. Existing runs are never overwritten.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    EXPERIMENTS_ROOT = Path("/content/drive/MyDrive") / DRIVE_EXPERIMENTS_DIR
else:
    EXPERIMENTS_ROOT = Path(os.getenv("BITETRACKER_EXPERIMENTS_DIR",
                                      Path.home() / "BiteTracker-experiments-local"))

EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)
previous_runs = sorted(p.name for p in EXPERIMENTS_ROOT.glob("run_*") if p.is_dir())
print(f"Experiment folder: {EXPERIMENTS_ROOT}")
print(f"Previous runs: {len(previous_runs)}" + (f" (latest: {previous_runs[-1]})" if previous_runs else ""))

## 5. Dataset source configuration

- **`food4education`** (default): reads the real Food4Education workbook live from its Google Sheets link through `src/google_sheet_loader.py`. Access is read-only and nothing is downloaded to disk. Add the share link as the Colab Secret **`BITETRACKER_GOOGLE_SHEET_URL`**. If the secret isn't set, the notebook asks for the link in a hidden input box. The sheet must be shared as *Anyone with the link → Viewer*.
- **`synthetic`**: uses the committed synthetic dataset `data/sample/synthetic_school_meals.csv`. It contains no real records; use it only to test that the notebook runs end to end.

Real-data results are **PROVISIONAL**. For Food4Education runs, row-level outputs (per-day predictions and actual-vs-predicted charts) are not saved to Drive by default.

In [ ]:
SYNTHETIC_PATH = WORK_DIR / "data" / "sample" / "synthetic_school_meals.csv"
SHEET_URL = None

if DATA_SOURCE == "food4education":
    SHEET_URL = _secret("BITETRACKER_GOOGLE_SHEET_URL")
    if not SHEET_URL:
        from getpass import getpass
        SHEET_URL = getpass("Paste the Food4Education Google Sheets share link (input is hidden): ").strip()
    if "/spreadsheets/d/" not in (SHEET_URL or ""):
        raise ValueError("That does not look like a Google Sheets link.")
    RESULTS_LABEL = et.PROVISIONAL_LABEL
    print("Food4Education Google Sheet link loaded (not shown).")
else:
    RESULTS_LABEL = et.SYNTHETIC_LABEL
    print(f"Synthetic dataset: {SYNTHETIC_PATH.relative_to(WORK_DIR)}")

INCLUDE_ROW_LEVEL = (DATA_SOURCE == "synthetic") if SAVE_ROW_LEVEL_OUTPUTS is None else bool(SAVE_ROW_LEVEL_OUTPUTS)
print(f"Save row-level predictions/charts to Drive: {INCLUDE_ROW_LEVEL}")
print(RESULTS_LABEL)

## 6. Dataset loading and validation

Loads the dataset with the project's own loader and checks it with `src/data_validation.py`. Only summary information is shown (counts, date range, quality score); individual records are never displayed.

In [ ]:
from src.data_loader import load_dataset
from src.data_validation import validate_upload

if DATA_SOURCE == "food4education":
    from src.google_sheet_loader import load_google_sheet_dataset
    df = load_google_sheet_dataset(SHEET_URL)
else:
    df = load_dataset(path=SYNTHETIC_PATH, generate_if_missing=False)

report = validate_upload(df)
dates = df["date"]
DATA_SUMMARY = {
    "source": DATA_SOURCE,
    "rows": int(len(df)),
    "schools": int(df["school_id"].nunique()),
    "date_range": [str(dates.min().date()), str(dates.max().date())],
    "columns": list(df.columns),
    "missing_values_by_column": {c: int(n) for c, n in df.isna().sum().items() if n},
    "target_meals_served": {k: round(float(v), 2) for k, v in df["meals_served"].describe().items()},
    "validation": {
        "is_valid": report.is_valid,
        "quality_score": report.quality_score,
        "forecast_readiness": report.forecast_readiness,
        "duplicate_rows": report.duplicate_rows,
        "errors": report.errors,
        "warnings": report.warnings,
    },
}
del df  # raw rows are not kept in the notebook session

print(f"Rows: {DATA_SUMMARY['rows']}  |  Schools: {DATA_SUMMARY['schools']}  |  "
      f"Dates: {DATA_SUMMARY['date_range'][0]} to {DATA_SUMMARY['date_range'][1]}")
print(f"Valid: {report.is_valid}  |  Quality score: {report.quality_score}/100  |  "
      f"Forecast readiness: {report.forecast_readiness}")
for message in report.errors:
    print("ERROR:", message)
for message in report.warnings:
    print("Warning:", message)
if not report.is_valid:
    raise RuntimeError("The dataset failed validation; training was not started.")

## 7. Training pipeline execution

Runs the project's existing one-command pipeline, `python scripts/train_models.py`. It loads the data, preprocesses it, engineers features, evaluates the naive baselines, tunes and cross-validates the four models, selects the model with the lowest TimeSeriesSplit validation MAE, and evaluates that model once on the held-out test period.

The pipeline writes its outputs inside the **temporary Colab clone**, not your local repository. Your local `models/` folder (including the model the Django app uses) is never touched. The full log is saved to the run folder as it is produced.

In [ ]:
# Git state is read before training: the pipeline rewrites tracked report files inside the clone.
GIT_INFO = {"repository": REPO, "requested_branch": BRANCH, "requested_commit": COMMIT or None,
            **et.git_info(WORK_DIR)}
RUN_DIR = et.create_run_dir(EXPERIMENTS_ROOT, STARTED_AT)
RUN_ID = RUN_DIR.name
(RUN_DIR / "logs" / "pip_install.log").write_text(PIP_LOG)
print(f"Run folder: {RUN_DIR}")

command = [sys.executable, "scripts/train_models.py"]
env = {**os.environ, "RANDOM_SEED": str(RANDOM_SEED), "PYTHONUNBUFFERED": "1"}
if DATA_SOURCE == "synthetic":
    env.pop("BITETRACKER_GOOGLE_SHEET_URL", None)
    command += ["--data-path", str(SYNTHETIC_PATH.relative_to(WORK_DIR))]
else:
    env["BITETRACKER_GOOGLE_SHEET_URL"] = SHEET_URL


def _scrub(line):
    return line.replace(SHEET_URL, "<google-sheet-link>") if SHEET_URL else line


def save_metadata(status, **extra):
    """metadata.json describes the run even if training fails part-way."""
    return et.write_json(RUN_DIR / "metadata.json", {
        "run_id": RUN_ID,
        "status": status,
        "results_label": RESULTS_LABEL,
        "data_source": DATA_SOURCE,
        "started_at_utc": STARTED_AT.isoformat(),
        "git": GIT_INFO,
        "config": {
            "random_seed": RANDOM_SEED,
            "command": "python scripts/train_models.py" + (
                f" --data-path {SYNTHETIC_PATH.relative_to(WORK_DIR)}" if DATA_SOURCE == "synthetic" else ""),
            "save_row_level_outputs": INCLUDE_ROW_LEVEL,
        },
        "environment": {**et.environment_info(), "google_colab": IN_COLAB},
        "dataset_summary": DATA_SUMMARY,
        **extra,
    })


save_metadata("running")
train_start = time.perf_counter()
with open(RUN_DIR / "logs" / "training.log", "w") as log:
    process = subprocess.Popen(command, cwd=WORK_DIR, env=env, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in process.stdout:
        line = _scrub(line)
        print(line, end="")
        log.write(line)
    TRAINING_EXIT_CODE = process.wait()
TRAINING_SECONDS = round(time.perf_counter() - train_start, 1)

if TRAINING_EXIT_CODE != 0:
    save_metadata("failed", training_exit_code=TRAINING_EXIT_CODE, training_seconds=TRAINING_SECONDS,
                  finished_at_utc=et.utc_now().isoformat())
    raise RuntimeError(f"Training failed (exit code {TRAINING_EXIT_CODE}). "
                       f"The log is saved in {RUN_DIR / 'logs' / 'training.log'}.")
print(f"\nTraining finished in {TRAINING_SECONDS}s.")

## 8. Model evaluation

These tables come from the CSV files the pipeline saved; nothing is recalculated here. The model is selected by **validation** MAE on the training period only. The test period is used once, after selection.

In [ ]:
import pandas as pd
from IPython.display import HTML, display

metrics_dir = WORK_DIR / "reports" / "metrics"
print(RESULTS_LABEL, "\n")


def show(title, filename):
    path = metrics_dir / filename
    print(title)
    if path.exists():
        display(pd.read_csv(path).round(3))
    else:
        print(f"  NOT AVAILABLE: {filename} was not produced by this run.")


show("Model selection (TimeSeriesSplit validation, training period only):", "validation_model_selection.csv")
show("All models and naive baselines on the held-out test period:", "model_comparison.csv")
show("Selected model vs naive baselines (test period):", "selected_model_vs_naive_baselines.csv")

cv_chart = WORK_DIR / "reports" / "figures" / "cv_score_comparison.html"
if cv_chart.exists():
    display(HTML(cv_chart.read_text()))

## 9. Saving experiment outputs

Copies the outputs into the run folder on Google Drive:

| Item | Saved as |
|---|---|
| Selected model | `model.joblib` (plus every model and its metadata in `models/`) |
| Metrics, baselines, model comparison | `metrics.json` and `metrics/*.csv` |
| Run details: Git commit SHA, settings, random seed, timestamps, package versions, dataset summary | `metadata.json`, `environment/pip_freeze.txt` |
| Charts (interactive HTML) | `plots/` |
| Generated reports | `reports/` |
| Logs | `logs/training.log`, `logs/pip_install.log` |

Any expected file the pipeline did not produce is listed as **missing**. Nothing is generated to fill the gap. Dataset rows (`data/processed/`) are never copied.

In [ ]:
manifest = et.collect_artifacts(WORK_DIR, RUN_DIR, include_row_level=INCLUDE_ROW_LEVEL)

metrics = et.build_metrics(WORK_DIR)
metrics = {"run_id": RUN_ID, "results_label": RESULTS_LABEL, "git_commit_sha": GIT_SHA, **metrics}
et.write_json(RUN_DIR / "metrics.json", metrics)

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(RUN_DIR / "environment").mkdir(exist_ok=True)
(RUN_DIR / "environment" / "pip_freeze.txt").write_text(freeze)

FINISHED_AT = et.utc_now()
save_metadata(
    "completed",
    finished_at_utc=FINISHED_AT.isoformat(),
    training_exit_code=TRAINING_EXIT_CODE,
    training_seconds=TRAINING_SECONDS,
    artifacts=manifest,
)

print(f"Saved {len(manifest['copied'])} files to {RUN_DIR}")
if manifest["missing"]:
    print("\nMISSING (expected but not produced by this run):")
    for item in manifest["missing"]:
        print("  -", item)
if manifest["skipped"]:
    print("\nNot saved on purpose:")
    for item in manifest["skipped"]:
        print("  -", item)

## 10. Experiment summary

In [ ]:
final = metrics.get("final_model", {})
print("=" * 72)
print(RESULTS_LABEL)
print("=" * 72)
print(f"Run ID:          {RUN_ID}")
print(f"Saved to:        {RUN_DIR}")
print(f"Git commit:      {GIT_SHA}  ({REPO} @ {BRANCH})")
print(f"Data source:     {DATA_SOURCE}  ({DATA_SUMMARY['rows']} rows, {DATA_SUMMARY['schools']} schools)")
print(f"Random seed:     {RANDOM_SEED}")
print(f"Duration:        {TRAINING_SECONDS}s training")
if final:
    print(f"Selected model:  {final.get('selected_model')} ({final.get('model_kind')})")
    print(f"Validation MAE:  {final.get('validation_mae'):.2f}  (selection criterion)")
    print(f"Test MAE / RMSE / R2: {final.get('final_test_mae'):.2f} / "
          f"{final.get('final_test_rmse'):.2f} / {final.get('final_test_r2'):.3f}")
else:
    print("Selected model:  NOT AVAILABLE (final_best_model_metadata.json was not produced)")
for row in (metrics["tables"].get("selected_model_vs_naive_baselines") or []):
    verdict = "beats" if row["Model Beats Baseline (MAE)"] else "does NOT beat"
    print(f"  {row['Selected Model']} {verdict} {row['Naive Baseline']} "
          f"(test MAE {row['Model Test MAE']:.2f} vs {row['Baseline Test MAE']:.2f})")
print(f"Missing artifacts: {manifest['missing'] or 'none'}")
if DATA_SOURCE == "food4education":
    print("\nPROVISIONAL: do not treat this model as the definitive real-data winner, and do not copy it "
          "into the local models/ folder used by the Django app.")

### Reviewing the results on your Mac

With **Google Drive for desktop** installed and signed in to the same Google account, this run appears at:

`~/Library/CloudStorage/GoogleDrive-<your-email>/My Drive/BiteTracker/experiments/<run_id>/`

From the project folder you can list every run:

```bash
python scripts/summarize_experiments.py
```

Open `metadata.json`, `metrics.json`, `logs/training.log` and the HTML charts in `plots/` in your IDE, or ask the AI agent to review them. See the README section *Google Colab and Google Drive workflow* for setup details.